# ODU Module 5 — Denoising Diffusion Model (Foster Chapter 8)
**Wahab / Open OnDemand JupyterLab | Oxford 102 Flowers | No Docker**

This is a **standalone, Foster-style adaptation**, not a verbatim copy of the repository notebook. It implements a noise-predicting U-Net, EMA weights, cosine noise schedule, MAE `n_loss`, and deterministic DDIM-style sampling. Run all cells in order on a GPU. **No experimental results are prefilled.**

**Required outputs:** `training_loss.png`, `samples_20.png`, `ablation_side_by_side.png`, `ablation_times.csv`, `run_metadata.json`, and a ZIP of outputs.

Reference: Foster, D. (2023). *Generative deep learning* (2nd ed., Chapter 8). O'Reilly. https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/tree/main/notebooks/08_diffusion/01_ddm

**Revision:** Corrected repeat/step accounting, checkpoint initialization, continuation epochs, and compiled sampling loop for high-step ablations. Run all cells from the top after restarting the kernel.


**Dataset prerequisite:** `archive.zip` must be uploaded separately; the notebook does not contain the dataset.


## 1. Configuration and environment

Launch a GPU-enabled Wahab JupyterLab session. Upload this notebook and `archive.zip`, then use **Run → Run All Cells**. The notebook automatically discovers the dataset.


In [1]:
import os, sys, time, json, random, platform, csv, zipfile
from pathlib import Path
SEED=42
EPOCHS=10                # Increase if samples are not recognizable; Foster default is 50
BATCH_SIZE=32            # Reduce to 16 or 8 for GPU out-of-memory errors
REPEATS_PER_EPOCH=5      # Foster-style dataset repeats per epoch
LEARNING_RATE=1e-4
IMAGE_SIZE=64
NUM_IMAGES=16
LOAD_MODEL=False         # Set True after at least one checkpoint is saved
OUT=Path('module5_ddm_outputs'); OUT.mkdir(exist_ok=True)
CKPT=OUT/'checkpoint.weights.h5'
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print('Python:',sys.version.split()[0], 'TensorFlow:',tf.__version__)
print('GPUs:',tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print('WARNING: GPU not detected. Request a GPU partition before training.')
print('Working directory:',Path.cwd().resolve())

2026-11-22 16:37:53.647565: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-22 16:37:53.683326: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-22 16:37:53.683363: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Python: 3.10.14 TensorFlow: 2.16.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Working directory: /home/lwalk051/Project Mid/Module 5 


## 2. Automatic dataset setup (`archive.zip`)

Upload `archive.zip` using the JupyterLab file browser. The next cell finds the archive in common Wahab directories, extracts nested images, and sets `DATA_DIR` automatically. **No path changes or edits to older notebook versions are required.**


In [2]:
# Automatic Oxford Flowers dataset setup — NO PATH EDITS REQUIRED.
# Upload archive.zip using JupyterLab's file browser. This cell searches common
# Wahab locations and nested directories, extracts safely, and verifies images.
from pathlib import Path
import zipfile, shutil, os

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
DATA_DIR = (Path.cwd() / 'flowers').resolve()

def image_files(folder):
    folder = Path(folder)
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS) if folder.is_dir() else []

def candidate_archives():
    roots = [Path.cwd(), Path.home(), Path.home() / 'Downloads',
             Path.home() / 'scratch', Path.home() / 'ondemand',
             Path(os.environ.get('SLURM_SUBMIT_DIR', str(Path.cwd())))]
    matches = []
    for root in roots:
        if not root.is_dir():
            continue
        for pattern in ('archive.zip', '*/archive.zip', '*/*/archive.zip'):
            try:
                matches.extend(root.glob(pattern))
            except (OSError, PermissionError):
                pass
    # Prefer files in the current notebook working directory, then newest.
    unique = {str(p.resolve()): p for p in matches if p.is_file()}
    return sorted(unique.values(), key=lambda p: (p.parent.resolve() != Path.cwd().resolve(), -p.stat().st_mtime))

print('Notebook working directory:', Path.cwd().resolve())
existing = image_files(DATA_DIR)
print('Already extracted images:', len(existing))
archives = candidate_archives()
print('Found archive.zip candidates:', [str(p) for p in archives[:10]])

if len(existing) < 100:
    if not archives:
        raise FileNotFoundError(
            'archive.zip was not found. In JupyterLab, use Upload Files to upload the '
            'Oxford Flowers Kaggle archive.zip into the folder containing this notebook. '
            'Then rerun this cell. Current working directory: ' + str(Path.cwd().resolve())
        )
    chosen = None
    for archive in archives:
        if not zipfile.is_zipfile(archive):
            print('Skipping invalid ZIP:', archive)
            continue
        with zipfile.ZipFile(archive) as z:
            members = [m for m in z.infolist() if not m.is_dir() and Path(m.filename).suffix.lower() in IMAGE_EXTS]
            if len(members) >= 100:
                chosen = archive
                print('Using:', archive.resolve(), '| images in ZIP:', len(members))
                print('Example paths:', [m.filename for m in members[:3]])
                DATA_DIR.mkdir(parents=True, exist_ok=True)
                for m in members:
                    # Prevent ZIP path traversal, including absolute paths and '..'.
                    dest = (DATA_DIR / m.filename).resolve()
                    if not dest.is_relative_to(DATA_DIR):
                        raise ValueError('Unsafe ZIP entry: ' + m.filename)
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    if not dest.is_file() or dest.stat().st_size != m.file_size:
                        with z.open(m) as reader, dest.open('wb') as writer:
                            shutil.copyfileobj(reader, writer)
                break
            else:
                print('Skipping ZIP with too few images:', archive, len(members))
    if chosen is None:
        raise ValueError('Found archive.zip but none contained at least 100 flower images. Verify the Kaggle download.')

found = image_files(DATA_DIR)
print('Dataset ready:', DATA_DIR)
print('Images found:', len(found))
print('First 3 images:', [str(p) for p in found[:3]])
if len(found) < 100:
    raise RuntimeError('Dataset extraction did not produce at least 100 images.')


Notebook working directory: /home/lwalk051/Project Mid/Module 5 
Already extracted images: 0
Found archive.zip candidates: ['/home/lwalk051/Project Mid/Module 5 /archive.zip', '/home/lwalk051/Option B/archive.zip']
Using: /home/lwalk051/Project Mid/Module 5 /archive.zip | images in ZIP: 8189
Example paths: ['dataset/test/image_00005.jpg', 'dataset/test/image_00006.jpg', 'dataset/test/image_00024.jpg']
Dataset ready: /home/lwalk051/Project Mid/Module 5 /flowers
Images found: 8189
First 3 images: ['/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00005.jpg', '/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00006.jpg', '/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00024.jpg']


In [3]:
# Recursively discover flower images, independent of Kaggle directory nesting.
paths=sorted(p for p in DATA_DIR.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png','.bmp')) if DATA_DIR.exists() else []
print('Image files:',len(paths)); print('First five:',[str(p) for p in paths[:5]])
if len(paths)<100:
    raise FileNotFoundError(f'Only {len(paths)} images found in {DATA_DIR.resolve()}; check dataset extraction.')
# Filter out unreadable files before training (no labels are required for unconditional diffusion).
valid=[]
for p in paths:
    try:
        with open(p,'rb') as f: tf.io.decode_image(f.read(),channels=3,expand_animations=False)
        valid.append(str(p))
    except Exception: pass
print('Valid images:',len(valid))
if len(valid)<100: raise RuntimeError('Too few decodable flower images')
def load_image(path):
    raw=tf.io.read_file(path)
    img=tf.io.decode_image(raw,channels=3,expand_animations=False)
    img.set_shape([None,None,3])
    img=tf.image.resize(img,[IMAGE_SIZE,IMAGE_SIZE],method='bilinear')
    return tf.cast(img,tf.float32)/255.0
train_ds=(tf.data.Dataset.from_tensor_slices(valid).shuffle(len(valid),seed=SEED,reshuffle_each_iteration=True)
          .map(load_image,num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE,drop_remainder=True)
          .prefetch(tf.data.AUTOTUNE))
steps_per_pass=len(valid)//BATCH_SIZE
steps_per_epoch=steps_per_pass*REPEATS_PER_EPOCH
print('Batch:',BATCH_SIZE,'Steps/epoch:',steps_per_epoch,'Epochs:',EPOCHS)
fig,axes=plt.subplots(2,4,figsize=(9,5))
for ax,img in zip(axes.flat,next(iter(train_ds))[:8]): ax.imshow(img.numpy());ax.axis('off')
plt.suptitle('Oxford 102 Flowers: training data check');plt.tight_layout();plt.show()

Image files: 8189
First five: ['/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00005.jpg', '/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00006.jpg', '/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00024.jpg', '/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00025.jpg', '/home/lwalk051/Project Mid/Module 5 /flowers/dataset/test/image_00029.jpg']
Valid images: 8189
Batch: 32 Steps/epoch: 1275 Epochs: 10


## 3. Foster-style noise prediction model
This compact U-Net retains the diffusion training objective and sampling workflow. The original repository architecture may differ in exact layer counts; describe this as an adaptation if you submit it. The target is Gaussian noise, and the optimized metric is **mean absolute error**, `n_loss`.

Compatibility fix: both downsampling layers explicitly set `pool_size=(2, 2)`, required by current Keras.

In [4]:
def sinusoidal_embedding(t,dim=64):
    half=dim//2
    frequencies=tf.exp(tf.linspace(tf.math.log(1.0),tf.math.log(1000.0),half))
    angles=2*np.pi*tf.cast(t,tf.float32)*frequencies
    return tf.concat([tf.sin(angles),tf.cos(angles)],axis=-1)

def residual_block(x,filters,emb):
    shortcut=x
    x=layers.Conv2D(filters,3,padding='same')(x)
    x=layers.GroupNormalization(groups=min(8,filters))(x)
    e=layers.Dense(filters)(layers.Activation('swish')(emb))
    x=layers.Add()([x,layers.Reshape((1,1,filters))(e)])
    x=layers.Activation('swish')(x)
    x=layers.Conv2D(filters,3,padding='same')(x)
    x=layers.GroupNormalization(groups=min(8,filters))(x)
    if shortcut.shape[-1]!=filters: shortcut=layers.Conv2D(filters,1)(shortcut)
    return layers.Add()([x,shortcut])

def make_unet():
    image=keras.Input((64,64,3)); noise_rate=keras.Input((1,))
    emb=layers.Lambda(lambda t: sinusoidal_embedding(t,64),output_shape=(64,))(noise_rate)
    emb=layers.Dense(128,activation='swish')(emb)
    x=layers.Conv2D(32,3,padding='same')(image)
    s1=residual_block(x,32,emb)
    x=layers.AveragePooling2D(pool_size=(2, 2))(s1)
    s2=residual_block(x,64,emb)
    x=layers.AveragePooling2D(pool_size=(2, 2))(s2)
    x=residual_block(x,128,emb)
    x=residual_block(x,128,emb)
    x=layers.UpSampling2D(interpolation='bilinear')(x)
    x=layers.Concatenate()([x,s2]);x=residual_block(x,64,emb)
    x=layers.UpSampling2D(interpolation='bilinear')(x)
    x=layers.Concatenate()([x,s1]);x=residual_block(x,32,emb)
    out=layers.Conv2D(3,1,kernel_initializer='zeros')(x)
    return keras.Model([image,noise_rate],out,name='foster_style_noise_unet')

# Cosine schedule as in Foster's diffusion approach.
def diffusion_schedule(t):
    start_angle=tf.acos(tf.constant(0.95,tf.float32))
    end_angle=tf.acos(tf.constant(0.02,tf.float32))
    angles=start_angle+tf.cast(t,tf.float32)*(end_angle-start_angle)
    return tf.sin(angles),tf.cos(angles) # noise_rate, signal_rate

class DDM(keras.Model):
    def __init__(self):
        super().__init__();self.network=make_unet();self.ema_network=make_unet()
        self.ema_network.set_weights(self.network.get_weights())
        self.loss_tracker=keras.metrics.Mean(name='n_loss')
    def call(self, inputs, training=False):
        return self.network(inputs,training=training)
    @property
    def metrics(self): return [self.loss_tracker]
    def train_step(self,images):
        if isinstance(images,tuple): images=images[0]
        n=tf.random.normal(tf.shape(images))
        t=tf.random.uniform((tf.shape(images)[0],1),minval=0,maxval=1)
        nr,sr=diffusion_schedule(t)
        nr=tf.reshape(nr,(-1,1,1,1));sr=tf.reshape(sr,(-1,1,1,1))
        noisy=sr*images+nr*n
        with tf.GradientTape() as tape:
            pred=self.network([noisy,tf.reshape(nr,(-1,1))],training=True)
            loss=tf.reduce_mean(tf.abs(n-pred))
        grads=tape.gradient(loss,self.network.trainable_weights)
        self.optimizer.apply_gradients(zip(grads,self.network.trainable_weights))
        for w,ew in zip(self.network.weights,self.ema_network.weights): ew.assign(0.999*ew+0.001*w)
        self.loss_tracker.update_state(loss)
        return {'n_loss':self.loss_tracker.result()}
    @tf.function(reduce_retracing=True)
    def denoise_step(self,x,t,next_t):
        nr,sr=diffusion_schedule(t)
        nr=tf.reshape(nr,(-1,1,1,1));sr=tf.reshape(sr,(-1,1,1,1))
        pred=self.ema_network([x,tf.reshape(nr,(-1,1))],training=False)
        predicted_image=(x-nr*pred)/tf.maximum(sr,1e-5)
        next_nr,next_sr=diffusion_schedule(next_t)
        new_x=tf.reshape(next_sr,(-1,1,1,1))*predicted_image+tf.reshape(next_nr,(-1,1,1,1))*pred
        return new_x,predicted_image
    def generate(self,num_images=16,diffusion_steps=20,initial_noise=None):
        x=(tf.random.normal((num_images,64,64,3)) if initial_noise is None else tf.convert_to_tensor(initial_noise,dtype=tf.float32))
        for step in range(diffusion_steps):
            t=tf.ones((num_images,1),dtype=tf.float32)*(1.0-step/diffusion_steps)
            next_t=tf.ones((num_images,1),dtype=tf.float32)*max(0.0,1.0-(step+1)/diffusion_steps)
            x,predicted_image=self.denoise_step(x,t,next_t)
        return tf.clip_by_value(predicted_image,0,1)

ddm=DDM();ddm.compile(optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE,weight_decay=1e-4))
# Build the subclassed model and EMA variables before loading Keras 3 weight checkpoints.
_ = ddm([tf.zeros((1,64,64,3)),tf.zeros((1,1))],training=False)
_ = ddm.ema_network([tf.zeros((1,64,64,3)),tf.zeros((1,1))],training=False)
print('Trainable parameters:',ddm.network.count_params())

Trainable parameters: 869827


## 4. Train, log MAE, and checkpoint
`LOAD_MODEL=True` reloads the most recent epoch checkpoint. To continue training, rerun this cell. The CSV history is appended after each epoch. Note that restarting the optimizer is not identical to uninterrupted training.

**Resume note:** `LOAD_MODEL=True` restores model/EMA weights, not optimizer state. Each resumed run adds `EPOCHS` epochs to the CSV history.


In [5]:
history_file=OUT/'training_history.csv'
previous_epochs=0
if LOAD_MODEL and history_file.exists():
    import pandas as pd
    previous_epochs=len(pd.read_csv(history_file))

class SaveEveryEpoch(keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs=None):
        ddm.save_weights(str(CKPT))
        with open(OUT/'training_history.csv','a',newline='') as f:
            writer=csv.writer(f)
            if f.tell()==0:writer.writerow(['epoch','n_loss','elapsed_seconds'])
            writer.writerow([epoch+1,float((logs or {}).get('n_loss',float('nan'))),time.perf_counter()-train_start])
        print('Saved checkpoint:',CKPT)
if LOAD_MODEL:
    if not CKPT.exists():raise FileNotFoundError(f'No checkpoint at {CKPT}')
    ddm.load_weights(str(CKPT));print('Loaded:',CKPT)
else:
    # Prevent mixing old and new histories.
    history_file.unlink(missing_ok=True)
    CKPT.unlink(missing_ok=True)
train_start=time.perf_counter()
history=ddm.fit(train_ds.repeat(),initial_epoch=previous_epochs,epochs=previous_epochs+EPOCHS,
                steps_per_epoch=steps_per_epoch,callbacks=[SaveEveryEpoch()],verbose=1)
training_seconds=time.perf_counter()-train_start
print('Training seconds this session:',round(training_seconds,1))

Epoch 1/10


I0000 00:00:1795383567.911527 1118523 service.cc:145] XLA service 0x7fcf70004790 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1795383567.911569 1118523 service.cc:153]   StreamExecutor device (0): Tesla V100-SXM2-16GB, Compute Capability 7.0
W0000 00:00:1795383569.461855 1118523 random_ops.cc:59] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. random_uniform/RandomUniform


   5/1275 ━━━━━━━━━━━━━━━━━━━━ 37s 29ms/step - n_loss: 0.7977  

I0000 00:00:1795383580.406803 1118523 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1275/1275 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - n_loss: 0.3237Saved checkpoint: module5_ddm_outputs/checkpoint.weights.h5
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 145s 93ms/step - n_loss: 0.3236
Epoch 2/10
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - n_loss: 0.1044Saved checkpoint: module5_ddm_outputs/checkpoint.weights.h5
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 119s 94ms/step - n_loss: 0.1044
Epoch 3/10
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - n_loss: 0.0976Saved checkpoint: module5_ddm_outputs/checkpoint.weights.h5
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 120s 94ms/step - n_loss: 0.0976
Epoch 4/10
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - n_loss: 0.0939Saved checkpoint: module5_ddm_outputs/checkpoint.weights.h5
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 120s 94ms/step - n_loss: 0.0939
Epoch 5/10
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - n_loss: 0.0920Saved checkpoint: module5_ddm_outputs/checkpoint.weights.h5
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 120s 94ms/step - n_loss: 0.0920
Epoch 6/10
1275/1275 ━━━━━━━━━━━━━━━━━━━━

In [6]:
import pandas as pd
hist=pd.read_csv(OUT/'training_history.csv')
plt.figure(figsize=(8,4));plt.plot(hist['epoch'],hist['n_loss'],marker='o')
plt.xlabel('Epoch');plt.ylabel('MAE noise prediction loss (n_loss)')
plt.title('DDM training loss (actual MAE, not MSE)');plt.grid(alpha=.3);plt.tight_layout()
plt.savefig(OUT/'training_loss.png',dpi=180);plt.show()
print(hist.tail().to_string(index=False))

 epoch   n_loss  elapsed_seconds
     6 0.090919       742.762797
     7 0.090665       862.447172
     8 0.089340       981.669671
     9 0.089418      1101.332264
    10 0.088874      1220.617895


## 5. Default sampling: 16 images, 20 steps
The 4×4 grid below uses the trained EMA model. Samples are *not* ground-truth flowers.

In [7]:
def show_grid(images,title,path=None):
    fig,axes=plt.subplots(4,4,figsize=(8,8))
    for i,ax in enumerate(axes.flat):ax.imshow(np.clip(images[i],0,1));ax.axis('off')
    fig.suptitle(title);fig.tight_layout()
    if path:fig.savefig(path,dpi=160,bbox_inches='tight')
    plt.show()
initial_noise=tf.random.stateless_normal((NUM_IMAGES,64,64,3),seed=(SEED,100))
t0=time.perf_counter()
default_samples=ddm.generate(num_images=16,diffusion_steps=20,initial_noise=initial_noise).numpy()
_ = float(tf.reduce_sum(default_samples).numpy())  # ensure GPU work is complete before timing
default_seconds=time.perf_counter()-t0
show_grid(default_samples,'20 denoising steps (default)',OUT/'samples_20.png')
print('20-step seconds/sample:',round(default_seconds/NUM_IMAGES,3))

20-step seconds/sample: 0.184


## 6. Required ablation: 50, 200, and 1000 steps
**Identical initial noise** is supplied to every condition. Generation runs sequentially. Time includes the sampling call, but not plotting. GPU warm-up and concurrent cluster load may affect timings.

In [8]:
ablation={};timings=[]
for steps in (50,200,1000):
    t0=time.perf_counter()
    generated=ddm.generate(num_images=16,diffusion_steps=steps,initial_noise=initial_noise).numpy()
    _ = float(tf.reduce_sum(generated).numpy())  # GPU synchronization
    elapsed=time.perf_counter()-t0
    ablation[steps]=generated
    timings.append({'sampling_steps':steps,'num_images':NUM_IMAGES,
                    'total_seconds':elapsed,'seconds_per_sample':elapsed/NUM_IMAGES})
    show_grid(generated,f'{steps} denoising steps',OUT/f'samples_{steps}.png')
    print(f'{steps:4d} steps: {elapsed:.2f} s total / {elapsed/NUM_IMAGES:.3f} s per sample')
pd.DataFrame(timings).to_csv(OUT/'ablation_times.csv',index=False)
display(pd.DataFrame(timings).round(3))

  50 steps: 0.33 s total / 0.021 s per sample
 200 steps: 1.29 s total / 0.080 s per sample
1000 steps: 6.36 s total / 0.397 s per sample


,sampling_steps,num_images,total_seconds,seconds_per_sample
0,50,16,0.328,0.021
1,200,16,1.286,0.080
2,1000,16,6.356,0.397


In [9]:
# Three side-by-side grids, each 4 rows x 4 columns, as required.
fig,axes=plt.subplots(4,12,figsize=(18,6.5))
for j,steps in enumerate((50,200,1000)):
    for k in range(16):
        ax=axes[k//4,j*4+k%4];ax.imshow(ablation[steps][k]);ax.axis('off')
    axes[0,j*4+1].set_title(f'{steps} steps',fontsize=14)
fig.suptitle('Matched initial noise — 50 vs 200 vs 1000 steps',y=1.02)
fig.tight_layout();fig.savefig(OUT/'ablation_side_by_side.png',dpi=170,bbox_inches='tight');plt.show()

## 7. Reproducibility metadata and output archive

In [10]:
metadata={'seed':SEED,'dataset':'nunenuh/pytorch-challange-flower-dataset (Oxford 102 Flowers)',
 'valid_images':len(valid),'image_size':IMAGE_SIZE,'epochs_requested_this_session':EPOCHS,'epochs_completed_total':previous_epochs+len(history.history['n_loss']),
 'batch_size':BATCH_SIZE,'repeats_per_epoch':REPEATS_PER_EPOCH,'steps_per_epoch':steps_per_epoch,
 'learning_rate':LEARNING_RATE,'loss':'MAE (n_loss)','tensorflow':tf.__version__,
 'python':sys.version.split()[0],'gpu_devices':[str(g) for g in tf.config.list_physical_devices('GPU')],
 'training_seconds_this_session':training_seconds,'checkpoint':str(CKPT),
 'default_20_steps_seconds_per_sample':default_seconds/NUM_IMAGES,
 'sampling_ablation':timings,'implementation':'Standalone Foster-style adaptation; not verbatim repository notebook'}
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
with zipfile.ZipFile('module5_ddm_outputs.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p.suffix!='.h5':z.write(p,arcname=p.name)
print('Saved: module5_ddm_outputs.zip')
print('Output files:',[p.name for p in OUT.iterdir()])

Saved: module5_ddm_outputs.zip
Output files: ['run_metadata.json', 'training_loss.png', 'samples_200.png', 'ablation_times.csv', 'checkpoint.weights.h5', 'ablation_side_by_side.png', 'samples_50.png', 'samples_1000.png', 'training_history.csv', 'samples_20.png']


## 8. Interpretation prompts for the 1–2 page PDF
**Training:** State actual epochs, batch size, GPU, number of images, final `n_loss`, and whether MAE decreased or plateaued. Never call it MSE.

**Default 20-step grid:** Describe recognizable flower structures, color, blur, artifacts, and diversity. Do not claim quality without looking at the actual grid.

**Ablation:** Compare matched image positions across 50, 200, and 1000 steps. Does extra computation improve petals, edges, or coherence? Does quality plateau? Quote exact seconds/sample from the CSV. The diffusion network predicts noise; reverse sampling progressively removes noise using the schedule.

**Earlier models:** VAE — Fashion-MNIST grayscale clothing, often smooth/blurry, explicit variational lower bound (ELBO), not exact data likelihood. GAN — LEGO bricks grayscale, often sharper but may show mode collapse, no tractable likelihood from standard GAN. Diffusion — Oxford 102 Flowers RGB, iterative sampling can provide diversity and detail, trained with a noise-prediction objective; this run does **not** compute exact likelihood. Different datasets prevent a controlled quantitative quality ranking.

**Disclosure example:** “I used ChatGPT to help adapt and document the Jupyter notebook and organize the report. I executed the code, inspected the generated images and timings, and verified the reported results.” Modify to match what you actually did.